# Context rot in a small local agent: does Deep Agents' context engineering help?

**Question:** a small local model works through a multi-step data task whose tool outputs overflow its context window. How much do Deep Agents' **offloading** and **summarisation** keep it accurate, compared with a plain tool-calling loop?

**Setup**
- **Data:** the [Our World in Data CO₂ dataset](https://github.com/owid/co2-data) (CC BY 4.0), pinned to one commit. A tool, `get_country_report(country)`, returns one country's 2004–2023 export for 12 emissions and energy metrics. That's a few thousand tokens per country, like the "pull a big export per company" example in LangChain's post.
- **Questions:** 24 generated questions with gold answers computed in pandas: 8 each that need **1, 3 or 5 country reports**. Every answer has the form `ANSWER: <country or year> | <number>`.
- **Model:** `qwen3.5:4b` through Ollama on a Colab T4, with a **16k-token context window**. One report fits easily, three are borderline and five overflow.
- **Arms:** the model, tool, system prompt, questions and decoding settings are the same in every arm.
  1. **Plain loop:** `langchain.agents.create_agent`, with no context management. When the window fills, Ollama silently drops the oldest tokens.
  2. **Deep Agents, techniques off:** the Deep Agents harness (filesystem tools and its prompt), with offloading and summarisation disabled.
  3. **+ Offloading:** large tool results are written to a file; the model sees a preview and reads the file with `read_file`/`grep`.
  4. **+ Offloading + summarisation:** older messages are condensed once the conversation reaches 85% of the window.
  5. **+ Offloading + early summarisation:** the same, but summarising at 50% of the window.
  4b, 5b. **The same two, untrimmed:** arms 4 and 5 used the summariser's constructor default, which lets the summary model read only 4,000 tokens of history. Beyond that, the history was silently replaced by a fixed "too long to summarize" note, with no model call. The "b" arms let the summariser read the whole history, as `create_deep_agent`'s own default does. Arms 4 and 5 stay in as a record of that pitfall. See [Summarisation's silent fallback](#summarisations-silent-fallback).
  6. *(optional, off by default)* **+ Researcher subagent:** each country lookup goes to a subagent with its own context, which returns only the numbers.
- **Part 2, summarisation:** a second task where the context grows from many small lookups that are never offloaded, so summarisation has to do the work. It compares no summary, summarising at 85% and at 50%, a summary prompt told to keep every number (each with the 4,000-token trim and untrimmed), and the agent-triggered `compact_conversation` tool, and checks each summary's numbers against the data. See [Part 2](#part-2-summarisation).

**TL;DR:** with `qwen3.5:4b` and a 16k window, once the context overflowed, a plain tool-calling loop got every answer wrong and gave no warning. Deep Agents' offloading recovered about half of those questions, and was cheap and safe. Summarisation helped too, but only after three non-default settings, and its most likely failure mode is silent: with one setting, it threw the history away without calling the model. The summaries never got a number wrong, but they dropped 16–34% of the values they needed to keep.

## Why this matters

Agents fail on long tasks less because the model is weak than because the context fills with tool output, old drafts and history. LangChain calls this *context rot*. [Deep Agents](https://github.com/langchain-ai/deepagents) is LangChain's open-source harness, and its [14 October 2026 post](https://blog.langchain.com/context-engineering-deep-agents-remember) describes five middleware techniques against context rot. This notebook tests the two that act inside a single run, **offloading** and **summarisation**, on deepagents **0.7.25** (14 October 2026).

Small local models have short windows, so they are where this should matter most. But Deep Agents' defaults are tuned for frontier models: tool results are offloaded only above about 20,000 tokens, and for a model that doesn't report its context size (every Ollama model), summarisation only starts at **170,000 tokens**. Out of the box, neither would ever fire here. This notebook therefore sets both thresholds explicitly for a 16k window and says so.

## Setup

On Colab this installs Ollama and the pinned Python packages. Locally it uses your own Ollama install (it must be running).

In [ ]:
import os, shutil, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
OLLAMA_VERSION = "0.40.4"

%pip install -q "deepagents==0.7.25" "langchain==1.4.6" "langchain-core==1.6.10" "langgraph==1.2.15" "langchain-ollama==1.1.0" tokenizers scipy

if IN_COLAB and shutil.which("ollama") is None:
    # The installer needs zstd to unpack the release; OLLAMA_VERSION pins the server version.
    subprocess.run("apt-get -qq install -y zstd > /dev/null", shell=True, check=True)
    subprocess.run(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh",
                   shell=True, check=True)

In [ ]:
import json, math, random, re, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import deepagents, langchain, langchain_core, langgraph, langchain_ollama
from deepagents import (FilesystemMiddleware, GeneralPurposeSubagentProfile, HarnessProfile,
                        create_deep_agent, register_harness_profile)
from deepagents.backends import StateBackend
from deepagents.middleware import (DEEPAGENTS_DEFAULT_SUMMARY_PROMPT, SummarizationMiddleware,
                                   SummarizationToolMiddleware)
from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessage, ToolMessage, messages_to_dict
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.errors import GraphRecursionError
from tokenizers import Tokenizer

SEED = 42
random.seed(SEED); np.random.seed(SEED)

from importlib.metadata import version
print({p: version(p) for p in ["deepagents", "langchain", "langchain-core", "langgraph", "langchain-ollama", "ollama"]})
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| local")

In [ ]:
OLLAMA_URL = "http://localhost:11434"


def ollama_up():
    try:
        urllib.request.urlopen(OLLAMA_URL, timeout=2)
        return True
    except OSError:
        return False


if not ollama_up() and shutil.which("ollama"):
    # One request at a time, and keep the model loaded between calls.
    env = {**os.environ, "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_KEEP_ALIVE": "-1"}
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
assert ollama_up(), "Ollama is not running. Locally, start the Ollama app or run `ollama serve`."
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())
if IN_COLAB:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())

## Config

In [ ]:
MODEL = "qwen3.5:4b"              # Ollama tag; "qwen3.5:9b" also fits a T4
TOKENIZER_ID = "Qwen/Qwen3.5-4B"  # the same model's tokenizer, used to count the tokens each call sends
NUM_CTX = 16384                   # the model's context window for every arm
NUM_PREDICT = 1024                # max new tokens per model call
THINK = False                     # Qwen3.5 thinking mode; on makes runs much slower
RECURSION_LIMIT = 80              # LangGraph steps per question (about 25-35 tool calls)

# Data: pinned to one commit of owid/co2-data
OWID_COMMIT = "382ee6c662b0ece26e111f263b44c029afad7787"
OWID_URL = f"https://raw.githubusercontent.com/owid/co2-data/{OWID_COMMIT}/owid-co2-data.csv"
YEARS = (2004, 2023)
METRICS = {   # column: unit, from the OWID codebook
    "co2": "million tonnes CO2",
    "co2_per_capita": "tonnes CO2 per person",
    "coal_co2": "million tonnes CO2",
    "oil_co2": "million tonnes CO2",
    "gas_co2": "million tonnes CO2",
    "cement_co2": "million tonnes CO2",
    "methane": "million tonnes CO2-equivalent",
    "nitrous_oxide": "million tonnes CO2-equivalent",
    "total_ghg": "million tonnes CO2-equivalent",
    "primary_energy_consumption": "TWh",
    "energy_per_capita": "kWh per person",
    "share_global_co2": "% of world CO2",
}
MIN_POPULATION = 10_000_000       # country pool: complete data for every metric and year, and at least this population
N_PER_LEVEL = 8                   # questions per level
LEVELS = (1, 3, 5)                # number of country reports a question needs
REL_TOL, ABS_TOL = 0.01, 0.01     # a number counts as correct within 1% (or 0.01 absolute)

# Context-engineering settings for a 16k window
OFFLOAD_TOKENS = 1000             # Deep Agents' unit: 1 "token" = 4 characters, so results over 4,000 chars are offloaded
SUMMARY_TRIGGER = int(0.85 * NUM_CTX)   # summarise at 85% of the window, counted with the real tokenizer
# Arm 5 summarises at 50% so it triggers more often on the longer runs.
SUMMARY_TRIGGER_EARLY = int(0.50 * NUM_CTX)
SUMMARY_KEEP_MESSAGES = 6         # recent messages kept verbatim after a summary (Deep Agents' default without a model profile)
# How much older history the summariser model may read. 4,000 is the default of the SummarizationMiddleware
# constructor, which the first runs used. Anything over it is trimmed from the start, and because the trim
# must begin at a user message, a longer history is dropped entirely and replaced by the fixed text
# "Previous conversation was too long to summarize." with no model call. create_deep_agent's own factory
# passes None (no trim). The "untrimmed" arms use None.
SUMMARY_TRIM_CONSTRUCTOR_DEFAULT = 4000

RUN_SUBAGENT_ARM = False          # optional 6th arm
RUN_PART1 = True                  # offloading experiment (arms 1-5b); cached runs are skipped either way
RUN_PART2 = True                  # summarisation experiment (arms S1-S5 and S2b-S4b), see Part 2

# Part 2: summarisation. Many small lookups instead of a few big reports, so nothing is offloaded
# and the context can only stay inside the window by being summarised.
LEVELS_2 = (6, 30, 50)            # countries per question: about 2k, 11k and 18k tokens of lookups
N_PER_LEVEL_2 = 6                 # questions per level
NUM_PREDICT_2 = 2048              # room for up to 50 parallel tool calls in one reply
RECURSION_LIMIT_2 = 300           # up to about 140 model/tool turns per question
COMPACT_MESSAGES = 40             # compact_conversation is allowed from 20 messages (half of this) onwards
SMOKE_N = None                    # e.g. 1: one question per level and a separate cache, for a quick end-to-end check

SYSTEM_PROMPT = (
    "You are a careful data analyst. Answer the user's question using the get_country_report tool, which returns "
    "one country's emissions and energy data, one value per line as `metric | year | value`. Look up every "
    "country the question names and never guess a number. Work out the answer from the exact values in the "
    "reports. When you are done, end your reply with exactly one line in this format:\n"
    "ANSWER: <country or year> | <number>"
)

USE_DRIVE = True                  # keep caches and traces on Google Drive so a disconnect resumes where it stopped
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/ML/2026-10-deep-agents-context-rot")
else:
    if USE_DRIVE:
        print("Not on Colab: USE_DRIVE ignored, caching under outputs/")
    BASE_DIR = Path("outputs")
CACHE_DIR = BASE_DIR / ("cache_smoke" if SMOKE_N else "cache")
TRACE_DIR, FIG_DIR = CACHE_DIR / "traces", Path("figures")
for d in (CACHE_DIR, TRACE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("cache:", CACHE_DIR)

## Model

The model and its tokenizer are shared by every arm. Decoding is greedy (`temperature=0`) with a fixed seed.

In [ ]:
subprocess.run(["ollama", "pull", MODEL], check=True, capture_output=True)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

llm = ChatOllama(model=MODEL, base_url=OLLAMA_URL, num_ctx=NUM_CTX, num_predict=NUM_PREDICT,
                 temperature=0, seed=SEED, reasoning=THINK, keep_alive=-1)
print(llm.invoke("Reply with the single word: ready").content)

TOK = Tokenizer.from_pretrained(TOKENIZER_ID)


def msg_text(m):
    content = m.content if isinstance(m.content, str) else json.dumps(m.content)
    calls = json.dumps(getattr(m, "tool_calls", None) or [])
    return f"{m.type}: {content} {calls if calls != '[]' else ''}"


def count_tokens(messages, *args, **kwargs):
    # Tokens the model receives for these messages, with the model's own tokenizer.
    # Leaves out the chat template and the tool schemas, so it slightly undercounts.
    return len(TOK.encode("\n".join(msg_text(m) for m in messages)).ids)

## Data

**Licence:** Our World in Data's CO₂ dataset is released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); the underlying sources are credited in its [codebook](https://github.com/owid/co2-data/blob/master/owid-co2-codebook.csv). Values are rounded for the reports (0 decimals from 1,000 upwards, 1 decimal from 10, otherwise 2), and the gold answers are computed from the same rounded values the agent sees.

In [ ]:
raw = pd.read_csv(OWID_URL)
y0, y1 = YEARS
d = raw[raw.iso_code.notna() & ~raw.iso_code.astype(str).str.startswith("OWID") & raw.year.between(y0, y1)]
cols = list(METRICS)
complete = d.groupby("country").apply(lambda g: len(g) == y1 - y0 + 1 and g[cols].notna().all().all(),
                                      include_groups=False)
big = d[d.year == y1].set_index("country").population >= MIN_POPULATION
POOL = sorted(c for c in complete[complete].index if big.get(c, False))
DATA = d[d.country.isin(POOL)].set_index(["country", "year"])[cols].sort_index()


def fmt(v):
    a = abs(v)
    return f"{v:.0f}" if a >= 1000 else f"{v:.1f}" if a >= 10 else f"{v:.2f}"


VALUES = DATA.map(lambda v: float(fmt(v)))   # the rounded values the reports show


def report(country):
    lines = [f"Country report: {country}, {y0}-{y1}. One value per line: metric | year | value.",
             "Units: " + "; ".join(f"{m} = {u}" for m, u in METRICS.items())]
    for m in cols:
        lines += [f"{m} | {y} | {fmt(DATA.loc[(country, y), m])}" for y in range(y0, y1 + 1)]
    return "\n".join(lines)


sample = report(POOL[0])
rep_tokens = len(TOK.encode(sample).ids)
print(f"{len(POOL)} countries in the pool, e.g. {', '.join(POOL[:8])} ...")
print(f"one report: {len(sample.splitlines())} lines, {len(sample):,} chars, {rep_tokens:,} Qwen tokens "
      f"(chars/4 would say {len(sample) // 4:,}). Window: {NUM_CTX:,} tokens")
for k in LEVELS:
    print(f"  {k} report(s) = about {k * rep_tokens:,} tokens -> {'fits' if k * rep_tokens < 0.5 * NUM_CTX else 'tight' if k * rep_tokens < NUM_CTX else 'overflows'}")
print("\n".join(sample.splitlines()[:5]), "\n...")

### Questions

Each level mixes two question types. Gold answers have a clear margin (at least 3% between the best and second-best candidate), so the 1% tolerance never decides between two near-tied candidates.

| Level | Type A (4 questions) | Type B (4 questions) |
|---|---|---|
| 1 country | Change in a metric between two years | Year in which a metric peaked, and its value |
| 3 or 5 countries | Country with the highest value in a given year | Country with the largest increase between two years |

In [ ]:
rng = random.Random(SEED)
M_LABEL = {m: m.replace("_", " ") for m in cols}


def margin_ok(vals, rel=0.03):
    top = sorted(vals, reverse=True)[:2]
    return len(top) < 2 or top[0] - top[1] > rel * max(abs(top[0]), 1e-9)


def make_question(k, kind):
    while True:
        m = rng.choice(cols)
        ya, yb = sorted(rng.sample(range(y0, y1 + 1), 2))
        if yb - ya < 5:
            continue
        cs = rng.sample(POOL, k)
        if k == 1:
            c = cs[0]
            if kind == "A":
                gold = round(VALUES.loc[(c, yb), m] - VALUES.loc[(c, ya), m], 4)
                if gold == 0:
                    continue
                q = (f"By how much did {c}'s {M_LABEL[m]} ({METRICS[m]}) change between {ya} and {yb}? "
                     f"Give {yb} minus {ya}, with the country as the name.")
                return dict(question=q, gold_entity=c, gold_value=gold)
            series = VALUES.loc[c, m]
            if not margin_ok(series.values):
                continue
            q = (f"In which year between {y0} and {y1} was {c}'s {M_LABEL[m]} ({METRICS[m]}) highest, "
                 f"and what was it? Give the year as the name.")
            return dict(question=q, gold_entity=str(int(series.idxmax())), gold_value=float(series.max()))
        names = ", ".join(cs[:-1]) + " and " + cs[-1]
        if kind == "A":
            vals = {c: VALUES.loc[(c, yb), m] for c in cs}
            q = f"Of {names}, which had the highest {M_LABEL[m]} ({METRICS[m]}) in {yb}, and what was the value?"
        else:
            vals = {c: VALUES.loc[(c, yb), m] - VALUES.loc[(c, ya), m] for c in cs}
            q = (f"Of {names}, which had the largest increase in {M_LABEL[m]} ({METRICS[m]}) between {ya} and "
                 f"{yb}, and how big was that increase ({yb} minus {ya})?")
        if not margin_ok(list(vals.values())):
            continue
        best = max(vals, key=vals.get)
        return dict(question=q, gold_entity=best, gold_value=round(float(vals[best]), 4))


rows = []
for k in LEVELS:
    for i in range(N_PER_LEVEL):
        kind = "A" if i < N_PER_LEVEL // 2 else "B"
        rows.append(dict(qid=f"k{k}_{i:02d}", k=k, kind=kind, **make_question(k, kind)))
QUESTIONS = pd.DataFrame(rows)
if SMOKE_N:
    QUESTIONS = QUESTIONS.groupby("k", group_keys=False).head(SMOKE_N).reset_index(drop=True)
QUESTIONS.to_csv("questions.csv", index=False)
print(QUESTIONS.shape)
for r in QUESTIONS.groupby("k").head(1).itertuples():
    print(f"[{r.qid}] {r.question}\n   gold: {r.gold_entity} | {r.gold_value}")

### The tool

The same tool object is used by every arm. It accepts a country name and is forgiving about case and spacing.

In [ ]:
POOL_LOOKUP = {c.lower(): c for c in POOL}


@tool
def get_country_report(country: str) -> str:
    """Return the full emissions and energy data export for one country (2004-2023), one value per line."""
    c = POOL_LOOKUP.get(country.strip().lower())
    if c is None:
        return f"No report for '{country}'. Available countries: {', '.join(POOL)}"
    return report(c)


print(get_country_report.invoke({"country": QUESTIONS.question[0].split("'s ")[0].split("did ")[-1]})[:300])

## Measurement

A callback records every model call: how many tokens were sent (counted with the Qwen tokenizer), the output tokens Ollama reports, latency, and whether tools were bound. Calls with no tools bound are the summariser's. A call **over the window** sent more tokens than `NUM_CTX`, so Ollama had to drop part of the conversation without saying so.

In [ ]:
class CallLog(BaseCallbackHandler):
    def __init__(self):
        self.calls, self._open = [], {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        params = kwargs.get("invocation_params") or {}
        rec = dict(t0=time.perf_counter(), n_msgs=len(messages[0]),
                   sent_tokens=count_tokens(messages[0]), has_tools=bool(params.get("tools")))
        self._open[run_id] = rec

    def on_llm_end(self, response, *, run_id, **kwargs):
        rec = self._open.pop(run_id, None)
        if rec is None:
            return
        msg = response.generations[0][0].message
        usage = getattr(msg, "usage_metadata", None) or {}
        rec.update(latency_s=time.perf_counter() - rec.pop("t0"), ollama_in=usage.get("input_tokens"),
                   out_tokens=usage.get("output_tokens"))
        if not rec["has_tools"]:   # summariser call: keep the summary it wrote
            rec["text"] = msg.content if isinstance(msg.content, str) else json.dumps(msg.content)
        self.calls.append(rec)

    def on_llm_error(self, error, *, run_id, **kwargs):
        self._open.pop(run_id, None)


ANSWER_RE = re.compile(r"ANSWER:\s*\**\s*(.+?)\s*\|\s*\**\s*([-+−]?[\d,]*\.?\d+(?:[eE][-+]?\d+)?)")


def parse_answer(text):
    found = ANSWER_RE.findall(text or "")
    if not found:
        return None, None
    entity, num = found[-1]
    try:
        return entity.strip(" *`'\"."), float(num.replace(",", "").replace("−", "-"))
    except ValueError:
        return entity.strip(), None


def score(q, entity, value):
    entity_ok = entity is not None and entity.strip().lower() == str(q.gold_entity).lower()
    value_ok = value is not None and abs(value - q.gold_value) <= max(REL_TOL * abs(q.gold_value), ABS_TOL)
    return bool(entity_ok), bool(value_ok)


print(parse_answer("so the answer is\nANSWER: Germany | -1,234.5"), score(QUESTIONS.iloc[0], *parse_answer(
    f"ANSWER: {QUESTIONS.gold_entity[0]} | {QUESTIONS.gold_value[0]}")))

## Baseline: a plain tool-calling loop

`create_agent` from LangChain 1.4 with the report tool and the system prompt, and nothing else. Every report stays in the conversation in full.

In [ ]:
def make_plain():
    return create_agent(llm, tools=[get_country_report], system_prompt=SYSTEM_PROMPT)

## New approach: Deep Agents with context engineering switched on step by step

Every Deep Agents arm uses the same harness: an in-memory `StateBackend` filesystem, the filesystem tools and Deep Agents' tool descriptions. A harness profile for Ollama models turns off the default general-purpose subagent (and the `execute` tool, which has no sandbox here), so arms 2–5b differ only in the two middlewares:

- `FilesystemMiddleware(tool_token_limit_before_evict=...)` controls **offloading**. `None` keeps every tool result inline; `OFFLOAD_TOKENS` writes larger results to `/large_tool_results/` and shows a preview.
- `SummarizationMiddleware(trigger=...)` controls **summarisation**. `None` never summarises; `("tokens", SUMMARY_TRIGGER)` summarises older messages once the conversation reaches 85% of the window. It counts tokens with the real tokenizer: Deep Agents' default estimate (4 characters per token) undercounts these number-heavy reports for Qwen (see the Data cell), so the trigger would fire too late. `trim_tokens_to_summarize` sets how much of the older history the summary model reads: arms 4 and 5 used the constructor's 4,000, arms 4b and 5b read all of it.

Passing a middleware with the same name to `create_deep_agent(middleware=[...])` replaces the built-in one in place.

In [ ]:
register_harness_profile("ollama", HarnessProfile(
    excluded_tools=frozenset({"execute"}),
    general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False),
))


def summarizer(trigger_tokens, backend, model=None, summary_prompt=None, trim=None):
    # trigger_tokens=None turns automatic summarisation off.
    # trim: tokens of older history the summary model may read (None = all of it); see SUMMARY_TRIM_CONSTRUCTOR_DEFAULT.
    model = model or llm
    if trigger_tokens is None:
        return SummarizationMiddleware(model=model, backend=backend, trigger=None)
    return SummarizationMiddleware(
        model=model, backend=backend, trigger=("tokens", trigger_tokens), keep=("messages", SUMMARY_KEEP_MESSAGES),
        token_counter=count_tokens, summary_prompt=summary_prompt or DEEPAGENTS_DEFAULT_SUMMARY_PROMPT,
        trim_tokens_to_summarize=trim,
        truncate_args_settings={"trigger": ("messages", 20), "keep": ("messages", 20)})


def make_deep(offload, summary_at=None, subagent=False, model=None, tools=None, system_prompt=None,
              summary_prompt=None, extra_middleware=(), trim=None):
    # Defaults are Part 1's model, tool and prompt; Part 2 passes its own.
    model = model or llm
    tools = [get_country_report] if tools is None else tools
    backend = StateBackend()
    middleware = [
        FilesystemMiddleware(backend=backend, tool_token_limit_before_evict=OFFLOAD_TOKENS if offload else None,
                             human_message_token_limit_before_evict=None),
        summarizer(summary_at, backend, model, summary_prompt, trim),
        *[make(backend) for make in extra_middleware],
    ]
    if not subagent:
        return create_deep_agent(model=model, tools=tools, system_prompt=system_prompt or SYSTEM_PROMPT,
                                 middleware=middleware, backend=backend)
    researcher = {
        "name": "country-researcher",
        "description": "Looks up one country's report and returns only the values you ask for.",
        "system_prompt": ("Call get_country_report for the country you are given, then reply with only the exact "
                          "values that were asked for, one per line as `metric | year | value`. Never return the raw report."),
        "tools": [get_country_report],
    }
    prompt = SYSTEM_PROMPT.replace("using the get_country_report tool, which returns",
                                   "by asking the country-researcher subagent (one country per task) for values from "
                                   "get_country_report, which returns")
    return create_deep_agent(model=llm, tools=[], system_prompt=prompt, subagents=[researcher],
                             middleware=middleware, backend=backend)


# Arm names are cache keys: renaming an arm reruns it.
TRIM_4K = SUMMARY_TRIM_CONSTRUCTOR_DEFAULT
ARMS = {
    "1 plain loop": make_plain,
    "2 deep agents, off": lambda: make_deep(offload=False),
    "3 + offloading": lambda: make_deep(offload=True),
    "4 + offload + summary": lambda: make_deep(offload=True, summary_at=SUMMARY_TRIGGER, trim=TRIM_4K),
    "5 + offload + early summary": lambda: make_deep(offload=True, summary_at=SUMMARY_TRIGGER_EARLY, trim=TRIM_4K),
    "4b + offload + summary, untrimmed": lambda: make_deep(offload=True, summary_at=SUMMARY_TRIGGER),
    "5b + offload + early summary, untrimmed": lambda: make_deep(offload=True, summary_at=SUMMARY_TRIGGER_EARLY),
}
if RUN_SUBAGENT_ARM:
    ARMS["6 + subagent"] = lambda: make_deep(offload=True, summary_at=SUMMARY_TRIGGER, subagent=True)
print(list(ARMS))

## Evaluation

Each (arm, question) runs once, and the result and the full message trace are cached, so a rerun after a disconnect skips what is done. In the Deep Agents arms, `state["messages"]` keeps the full raw history even after summarisation (the summary is applied per model call), so tool usage can be counted from it.

In [ ]:
def slug(arm):
    return re.sub(r"[^a-z0-9]+", "_", arm.lower()).strip("_")


def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {rec["qid"]: rec for rec in map(json.loads, f)}


def final_text(messages):
    for m in reversed(messages):
        if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
            return m.content
    return ""


def tool_counts(messages):
    names = [tc["name"] for m in messages if isinstance(m, AIMessage) for tc in (m.tool_calls or [])]
    return {n: names.count(n) for n in sorted(set(names))}


def run_arm(arm, arms=None, questions=None, recursion_limit=None):
    # Defaults are Part 1's arms and questions; Part 2 passes its own.
    arms = ARMS if arms is None else arms
    questions = QUESTIONS if questions is None else questions
    recursion_limit = recursion_limit or RECURSION_LIMIT
    path = CACHE_DIR / f"{slug(arm)}.jsonl"
    done = load_cache(path)
    todo = questions[~questions.qid.isin(done)]
    print(f"{arm}: {len(done)} cached, {len(todo)} to run")
    if not len(todo):
        return
    agent = arms[arm]()
    (TRACE_DIR / slug(arm)).mkdir(exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for q in tqdm(list(todo.itertuples()), desc=arm):
            log, t0, status, messages = CallLog(), time.perf_counter(), "ok", []
            try:
                state = agent.invoke({"messages": [{"role": "user", "content": q.question}]},
                                     config={"recursion_limit": recursion_limit, "callbacks": [log],
                                             "configurable": {"thread_id": f"{slug(arm)}-{q.qid}"}})
                messages = state["messages"]
            except GraphRecursionError:
                status = "step_limit"
            except Exception as e:  # keep going; the error is recorded and counts as wrong
                status = f"error: {type(e).__name__}: {e}"[:300]
            wall = time.perf_counter() - t0
            text = final_text(messages)
            entity, value = parse_answer(text)
            entity_ok, value_ok = score(q, entity, value)
            calls = pd.DataFrame(log.calls)
            agent_calls = calls[calls.has_tools] if len(calls) else calls
            rec = dict(qid=q.qid, k=int(q.k), kind=q.kind, arm=arm, status=status, entity=entity, value=value,
                       entity_ok=entity_ok, value_ok=value_ok, correct=entity_ok and value_ok, wall_s=wall,
                       n_calls=len(agent_calls), n_summary_calls=int((~calls.has_tools).sum()) if len(calls) else 0,
                       peak_sent=int(calls.sent_tokens.max()) if len(calls) else 0,
                       total_sent=int(calls.sent_tokens.sum()) if len(calls) else 0,
                       out_tokens=int(calls.out_tokens.fillna(0).sum()) if len(calls) else 0,
                       n_over_window=int((calls.sent_tokens > NUM_CTX).sum()) if len(calls) else 0,
                       tools=tool_counts(messages), answer_text=text[-600:])
            f.write(json.dumps(rec) + "\n"); f.flush()
            (TRACE_DIR / slug(arm) / f"{q.qid}.json").write_text(json.dumps(
                {"calls": log.calls, "messages": messages_to_dict(messages)}, default=str))


if RUN_PART1:
    for arm in ARMS:
        run_arm(arm)

In [ ]:
res = pd.concat([pd.DataFrame(load_cache(CACHE_DIR / f"{slug(a)}.jsonl").values()) for a in ARMS], ignore_index=True)
res = res[res.qid.isin(QUESTIONS.qid)]
res["over_window"] = res.n_over_window > 0
for name in ["read_file", "grep", "get_country_report", "task"]:
    res[f"n_{name}"] = res.tools.map(lambda t: t.get(name, 0))


def compactions(arm, qid):
    # Automatic summarisations, read from the agent's calls: without one, every agent call sees at least two
    # more messages than the last (the reply and its tool result), so a call that sees no more means older
    # messages were replaced by a summary. A "fallback" is one with no summariser model call just before it:
    # the summary was the fixed "too long to summarize" text and the history it replaced was lost.
    t = json.loads((TRACE_DIR / slug(arm) / f"{qid}.json").read_text())
    n = fallback = 0
    prev, summarised = None, False
    for c in t["calls"]:
        if not c["has_tools"]:
            summarised = True
            continue
        if prev is not None and c["n_msgs"] <= prev:
            n += 1
            fallback += not summarised
        prev, summarised = c["n_msgs"], False
    return n, fallback


res[["n_auto_compactions", "n_fallbacks"]] = [compactions(r.arm, r.qid) for r in res.itertuples()]

# Two scores. Strict: the reply must contain the `ANSWER: <name> | <number>` line.
# Lenient: also correct if the reply's last line names the gold country/year and contains the gold number
# (within the same tolerance), ignoring years quoted in the question. The first full run showed the
# Deep Agents arms often stated the right answer in prose and dropped the ANSWER line, so strict scoring
# measures format-following as much as accuracy. Lenient accuracy is the headline metric below.
NUM_RE = re.compile(r"[-−]?\d[\d,]*\.?\d*")


def lenient_ok(r, questions=None):
    if r.correct:
        return True
    q = (QUESTIONS if questions is None else questions).set_index("qid").loc[r.qid]
    lines = [s.strip() for s in (r.answer_text or "").splitlines() if s.strip()]
    last = re.sub(r"[*`]", "", lines[-1]) if lines else ""
    years = set(re.findall(r"\b(?:19|20)\d\d\b", q.question))
    nums = [float(n.replace(",", "").replace("−", "-")) for n in NUM_RE.findall(last) if n not in years]
    tol = max(REL_TOL * abs(q.gold_value), ABS_TOL)
    return str(q.gold_entity).lower() in last.lower() and any(abs(n - q.gold_value) <= tol for n in nums)


res = res.rename(columns={"correct": "correct_strict"}).assign(correct=lambda d: d.correct_strict)
res["correct_lenient"] = res.apply(lenient_ok, axis=1)
res["format_miss"] = res.correct_lenient & ~res.correct_strict
res.drop(columns=["tools", "correct"]).to_csv("results.csv", index=False)
print(res.groupby("arm").status.value_counts().unstack(fill_value=0))

# Did summarisation actually fire? A summariser call is a model call with no tools bound. Fallbacks replace
# history without any model call, so they are counted from the traces instead (see compactions()).
fired = res.groupby("arm").agg(runs_with_summary=("n_summary_calls", lambda s: int((s > 0).sum())),
                               summary_calls=("n_summary_calls", "sum"),
                               compactions=("n_auto_compactions", "sum"), fallbacks=("n_fallbacks", "sum"),
                               max_peak_sent=("peak_sent", "max")).reindex(list(ARMS))
display(fired)

## Results

In [ ]:
from scipy.stats import binomtest


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    centre, half = (p + z * z / (2 * n)) / (1 + z * z / n), z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


g = res.groupby("arm")
summary = pd.DataFrame({
    "accuracy": g.correct_lenient.mean(),
    "acc_95ci": [f"{lo:.2f}-{hi:.2f}" for lo, hi in (wilson(d.correct_lenient.sum(), len(d)) for _, d in g)],
    "accuracy_strict": g.correct_strict.mean(),
    "format_misses": g.format_miss.sum(),
    "runs_over_window": g.over_window.mean(),
    "peak_sent_median": g.peak_sent.median(),
    "calls_mean": g.n_calls.mean(),
    "summaries_mean": g.n_summary_calls.mean(),
    "compactions_mean": g.n_auto_compactions.mean(),
    "fallbacks_mean": g.n_fallbacks.mean(),
    "read_file_mean": g.n_read_file.mean(),
    "grep_mean": g.n_grep.mean(),
    "wall_s_median": g.wall_s.median(),
    "no_answer": g.entity.apply(lambda s: s.isna().mean()),
}).reindex(list(ARMS)).round(3)
summary.to_csv("summary.csv")
by_level = pd.concat([
    res.pivot_table(index="arm", columns="k", values=col, aggfunc="mean").round(3).rename(columns=lambda k: f"{name}_{k}_countries")
    for col, name in [("correct_lenient", "acc"), ("correct_strict", "strict")]], axis=1)
by_level.to_csv("accuracy_by_level.csv")
display(summary)
display(by_level)

# Paired exact McNemar test of each arm against the plain loop (same questions, lenient scores).
# Where arm 4 never compacted, it sent the model the same thing as arm 3, so their disagreements on those
# questions show run-to-run noise (Ollama isn't fully deterministic even at temperature 0).
base = res[res.arm == list(ARMS)[0]].set_index("qid").correct_lenient
for arm in list(ARMS)[1:]:
    other = res[res.arm == arm].set_index("qid").correct_lenient.reindex(base.index)
    b, c = int((base & ~other).sum()), int((~base & other).sum())
    p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    print(f"{arm} vs plain: wins {c}, losses {b}, exact McNemar p = {p:.3f}")
a3, a4 = (res[res.arm == a].set_index("qid") for a in ["3 + offloading", "4 + offload + summary"])
same = a4.index[a4.n_auto_compactions == 0]
print("noise check, arm 3 vs arm 4 disagreements where arm 4 never compacted:",
      [q for q in same if a3.correct_lenient.get(q) != a4.correct_lenient[q]], f"(of {len(same)} questions)")

In [ ]:
colors = dict(zip(ARMS, plt.cm.viridis(np.linspace(0.05, 0.85, len(ARMS)))))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
for arm in ARMS:
    acc = res[res.arm == arm].groupby("k").correct_lenient.mean()
    ax.plot(acc.index, acc.values, marker="o", label=arm, color=colors[arm])
ax.set(xticks=list(LEVELS), xlabel="Country reports the question needs", ylabel="Accuracy (lenient)", ylim=(-0.05, 1.05),
       title="Accuracy as the task's context load grows")
ax.legend(fontsize=8)
ax = axes[1]
w = 0.8 / len(ARMS)
for i, arm in enumerate(ARMS):
    peak = res[res.arm == arm].groupby("k").peak_sent.median()
    ax.bar(np.arange(len(LEVELS)) + i * w - 0.4 + w / 2, peak.reindex(LEVELS).values, w, label=arm, color=colors[arm])
ax.axhline(NUM_CTX, color="crimson", ls="--", lw=1)
ax.text(len(LEVELS) - 0.5, NUM_CTX, f" window {NUM_CTX:,}", color="crimson", va="bottom", ha="right", fontsize=8)
ax.set(xticks=range(len(LEVELS)), xticklabels=LEVELS, xlabel="Country reports the question needs",
       ylabel="Median peak tokens sent in one call", title="Largest context the model was sent")
fig.tight_layout(); fig.savefig(FIG_DIR / "accuracy_and_context.png", dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.boxplot([res[res.arm == a].wall_s for a in ARMS], showfliers=True)
ax.set_xticks(range(1, len(ARMS) + 1), [a.replace(" + ", "\n+ ") for a in ARMS], fontsize=8)
ax.set(ylabel="Seconds per question", title=f"Wall time per question ({MODEL}, T4)")
fig.tight_layout(); fig.savefig(FIG_DIR / "wall_time.png", dpi=160); plt.show()

## Qualitative examples

Side-by-side final answers for a few questions, then a condensed trace (tool calls and the size of each call) for one question that arm 4 got wrong.

In [ ]:
pd.set_option("display.max_colwidth", 120)
pick = QUESTIONS.groupby("k").head(1).qid.tolist() + QUESTIONS[QUESTIONS.k == max(LEVELS)].qid.tolist()[1:3]
side = res[res.qid.isin(pick)].pivot_table(index="qid", columns="arm", values="answer_text",
                                           aggfunc=lambda s: (s.iloc[0] or "")[-160:])
side.insert(0, "gold", QUESTIONS.set_index("qid").loc[side.index].apply(lambda r: f"{r.gold_entity} | {r.gold_value}", axis=1))
display(side)


def show_trace(arm, qid, width=160):
    t = json.loads((TRACE_DIR / slug(arm) / f"{qid}.json").read_text())
    print(f"--- {arm} / {qid}: {QUESTIONS.set_index('qid').question[qid]}")
    for c in t["calls"]:
        print(f"  model call: sent {c['sent_tokens']:,} tokens{' (OVER WINDOW)' if c['sent_tokens'] > NUM_CTX else ''}"
              f"{'' if c['has_tools'] else ' [summariser]'}")
    for m in t["messages"]:
        data = m["data"]
        for tc in data.get("tool_calls") or []:
            print(f"  -> {tc['name']}({json.dumps(tc['args'])[:100]})")
        if m["type"] == "tool":
            print(f"  <- {str(data['content'])[:width].replace(chr(10), ' / ')}")
    print("  final:", (res.set_index(["arm", "qid"]).answer_text.get((arm, qid)) or "")[-width:].replace("\n", " / "))


arm4 = "4 + offload + summary"
fails = res[(res.arm == arm4) & ~res.correct_lenient].sort_values("k", ascending=False)
if len(fails):
    show_trace(arm4, fails.qid.iloc[0])
    show_trace("1 plain loop", fails.qid.iloc[0])

# One run where arm 5b's summariser fired, to see what the summary kept and what the agent did next
arm5 = "5b + offload + early summary, untrimmed"
fired5 = res[(res.arm == arm5) & (res.n_summary_calls > 0)].sort_values("k", ascending=False)
if len(fired5):
    q5 = fired5.qid.iloc[0]
    show_trace(arm5, q5)
    t = json.loads((TRACE_DIR / slug(arm5) / f"{q5}.json").read_text())
    summ = [c.get("text") for c in t["calls"] if not c["has_tools"] and c.get("text")]
    print("\n  summary the agent continued from (first 800 chars):\n ", (summ[0] if summ else "not recorded")[:800])
else:
    print(f"{arm5}: the summariser never fired")

## Summarisation's silent fallback

Arms 4 and 5 (and S2–S4 in Part 2) build `SummarizationMiddleware` directly, and its constructor lets the summary model read at most `trim_tokens_to_summarize=4000` tokens of older history. LangChain trims longer history from the start, and the trimmed part must begin with a user message. In a single-question run the only user message is the question at the top, so any history over 4,000 tokens is trimmed to nothing. The middleware then skips the model call and uses the fixed text *"Previous conversation was too long to summarize."* as the summary: the agent loses everything before the kept messages, and no summariser call shows up in a callback log. `create_deep_agent`'s built-in summariser passes `None` instead, so the agent you get from Deep Agents' own defaults doesn't hit this. The "b" arms do the same.

The table counts compactions per arm, and how many were fallbacks.

In [ ]:
display(fired)
fb = res[res.n_fallbacks > 0].sort_values(["arm", "k"])
print(f"{len(fb)} runs had at least one fallback:", ", ".join(f"{r.arm} / {r.qid}" for r in fb.itertuples()) or "none")
if len(fb):
    show_trace(fb.arm.iloc[-1], fb.qid.iloc[-1])

# Part 2: summarisation

Part 1 could barely test summarisation. Offloading moved each big report into a file, so the context rarely grew far enough for the summariser to fire, and when it did, it mostly hit the silent fallback above. To test summarisation directly, Part 2 makes the context grow from **many small results**, none big enough to offload. The only way to stay inside the window is then to compress the history.

- **Tool:** `get_series(country, metric)` returns one country's 2004–2023 values for one metric, about 370 tokens. It's well under the offload threshold, so offloading never applies.
- **Questions:** "Of these *N* countries, which had the highest *metric* in *year*?" and "...the largest increase between *year 1* and *year 2*?" With *N* = 6, 30 or 50 countries, that's about 2k, 11k and 18k tokens of lookups. That's below both triggers, between the 50% and 85% triggers, and over the window. Gold answers come from the same rounded values the tool returns.
- **Arms:** the same Deep Agents harness, model and prompt in every arm; only the summarisation setup changes.
  - **S1 No summarisation:** shows what overflow does to this task.
  - **S2 Summarise at 85%** (13.9k tokens): the same trigger as Part 1's arm 4.
  - **S3 Summarise at 50%** (8.2k tokens): earlier and more often.
  - **S4 85% + keep-the-numbers prompt:** Deep Agents' default summary prompt plus one instruction to list every value looked up so far, verbatim. LangChain's post recommends customising the summary prompt, and Part 1 showed why: a default summary invented a proxy argument and recorded a wrong value.
  - **S5 `compact_conversation` tool:** no automatic trigger. The model decides when to compact by calling Deep Agents' compaction tool. The tool refuses until the conversation is half-way to its threshold (20 messages here), and a short system-prompt nudge tells the model the tool exists. Without a nudge, Deep Agents adds none.
  - **S2b, S3b, S4b untrimmed:** S2–S4 again, with the summary model reading the whole older history. S2–S4 used the constructor's 4,000-token trim (see [Summarisation's silent fallback](#summarisations-silent-fallback)), so in the first run every compaction was either a fallback or a summary of the question alone: the summariser never saw a lookup.
- **Number fidelity:** for every summary the runs produce, the notebook checks, for each country the question needs whose series was in the history the summary replaced, whether the summary kept a needed value correctly, stated a different number, or dropped it. A later summary is held to every lookup before it, so it must carry forward what earlier ones kept. Only numbers written right after the country's name (up to the next country name or the end of the line) count, compared as numbers, so 9.3 matches 9.30. It's a heuristic.

In [ ]:
@tool
def get_series(country: str, metric: str) -> str:
    """Return one country's yearly values (2004-2023) for one metric, one line per year as `metric | year | value`.
    Metrics: co2, co2_per_capita, coal_co2, oil_co2, gas_co2, cement_co2, methane, nitrous_oxide, total_ghg,
    primary_energy_consumption, energy_per_capita, share_global_co2."""
    c = POOL_LOOKUP.get(country.strip().lower())
    m = metric.strip().lower().replace(" ", "_")
    if c is None:
        return f"No data for '{country}'. Available countries: {', '.join(POOL)}"
    if m not in METRICS:
        return f"Unknown metric '{metric}'. Metrics: {', '.join(cols)}"
    return f"{c} | {m} | {y0}-{y1} ({METRICS[m]})\n" + "\n".join(
        f"{m} | {y} | {fmt(DATA.loc[(c, y), m])}" for y in range(y0, y1 + 1))


SYSTEM_PROMPT_2 = (
    "You are a careful data analyst. Answer the user's question using the get_series tool, which returns one "
    "country's yearly values for one metric. Look up every country the question names and never guess a number. "
    "Work out the answer from the exact values the tool returns. When you are done, end your reply with exactly "
    "one line in this format:\nANSWER: <country> | <number>"
)

rng2 = random.Random(SEED + 1)   # separate stream, so Part 1's questions are unchanged


def make_question_2(n, kind):
    while True:
        m = rng2.choice(cols)
        ya, yb = sorted(rng2.sample(range(y0, y1 + 1), 2))
        if yb - ya < 5:
            continue
        cs = rng2.sample(POOL, n)
        if kind == "A":
            vals = {c: VALUES.loc[(c, yb), m] for c in cs}
            needed = {c: [fmt(DATA.loc[(c, yb), m])] for c in cs}
            q = f"Of {', '.join(cs[:-1])} and {cs[-1]}, which had the highest {M_LABEL[m]} ({METRICS[m]}) in {yb}, and what was the value?"
        else:
            vals = {c: VALUES.loc[(c, yb), m] - VALUES.loc[(c, ya), m] for c in cs}
            needed = {c: [fmt(DATA.loc[(c, ya), m]), fmt(DATA.loc[(c, yb), m]), fmt(round(vals[c], 4))] for c in cs}
            q = (f"Of {', '.join(cs[:-1])} and {cs[-1]}, which had the largest increase in {M_LABEL[m]} ({METRICS[m]}) "
                 f"between {ya} and {yb}, and how big was that increase ({yb} minus {ya})?")
        if not margin_ok(list(vals.values())):
            continue
        best = max(vals, key=vals.get)
        return dict(question=q, gold_entity=best, gold_value=round(float(vals[best]), 4), needed=json.dumps(needed))


rows = []
for n in LEVELS_2:
    for i in range(N_PER_LEVEL_2):
        kind = "A" if i < N_PER_LEVEL_2 // 2 else "B"
        rows.append(dict(qid=f"s{n}_{i:02d}", k=n, kind=kind, **make_question_2(n, kind)))
QUESTIONS_2 = pd.DataFrame(rows)
if SMOKE_N:
    QUESTIONS_2 = QUESTIONS_2.groupby("k", group_keys=False).head(SMOKE_N).reset_index(drop=True)
QUESTIONS_2.drop(columns=["needed"]).to_csv("questions_part2.csv", index=False)

one = get_series.invoke({"country": POOL[0], "metric": "coal_co2"})
per_lookup = len(TOK.encode(one).ids) + 30   # result plus the tool call that asked for it
print(f"{len(QUESTIONS_2)} questions; one lookup is about {per_lookup} tokens")
for n in LEVELS_2:
    print(f"  {n} countries = about {n * per_lookup:,} tokens of lookups (window {NUM_CTX:,}; triggers "
          f"{SUMMARY_TRIGGER_EARLY:,} and {SUMMARY_TRIGGER:,})")
print(one[:120], "...")

### Part 2 arms

The keep-the-numbers prompt is Deep Agents' default summary prompt with one extra block, inserted just before the conversation it summarises. In S5, automatic summarisation is off, and a second summariser serves only as the engine behind the `compact_conversation` tool. Its trigger (85% of the window, or 40 messages) only sets when the tool is allowed to run: from half of either.

In [ ]:
KEEP_NUMBERS = (
    "<exact_values>\n"
    "This task depends on exact numbers. In the SUMMARY section, list every value that has been looked up so far, "
    "one per line, exactly as `country | metric | year | value`, copied verbatim from the tool results. Never round, "
    "estimate or infer a value, and never use one metric as a stand-in for another. If a country has not been "
    "looked up yet, say so.\n"
    "</exact_values>\n"
)
assert "\n<messages>\n" in DEEPAGENTS_DEFAULT_SUMMARY_PROMPT
SUMMARY_PROMPT_KEEP = DEEPAGENTS_DEFAULT_SUMMARY_PROMPT.replace("\n<messages>\n", "\n" + KEEP_NUMBERS + "\n<messages>\n", 1)

COMPACT_NUDGE = (
    "You have a compact_conversation tool. Your context window is only about 16,000 tokens and each lookup uses "
    "about 300 of them. When the conversation is getting long and you still have lookups to do, call "
    "compact_conversation to replace older messages with a summary and free up space."
)

llm2 = ChatOllama(model=MODEL, base_url=OLLAMA_URL, num_ctx=NUM_CTX, num_predict=NUM_PREDICT_2,
                  temperature=0, seed=SEED, reasoning=THINK, keep_alive=-1)


def compact_tool(backend):
    # Constructor-default trim, as in the first run; the model never called the tool there.
    engine = SummarizationMiddleware(
        model=llm2, backend=backend, keep=("messages", SUMMARY_KEEP_MESSAGES), token_counter=count_tokens,
        trigger=[("tokens", SUMMARY_TRIGGER), ("messages", COMPACT_MESSAGES)], trim_tokens_to_summarize=TRIM_4K)
    return SummarizationToolMiddleware(engine, system_prompt=COMPACT_NUDGE)


part2 = dict(model=llm2, tools=[get_series], system_prompt=SYSTEM_PROMPT_2, offload=False)
ARMS_2 = {
    "S1 no summary": lambda: make_deep(**part2),
    "S2 summary at 85%": lambda: make_deep(**part2, summary_at=SUMMARY_TRIGGER, trim=TRIM_4K),
    "S3 summary at 50%": lambda: make_deep(**part2, summary_at=SUMMARY_TRIGGER_EARLY, trim=TRIM_4K),
    "S4 85% + keep-numbers prompt": lambda: make_deep(**part2, summary_at=SUMMARY_TRIGGER, trim=TRIM_4K,
                                                      summary_prompt=SUMMARY_PROMPT_KEEP),
    "S5 compact_conversation tool": lambda: make_deep(**part2, extra_middleware=[compact_tool]),
    "S2b 85%, untrimmed": lambda: make_deep(**part2, summary_at=SUMMARY_TRIGGER),
    "S3b 50%, untrimmed": lambda: make_deep(**part2, summary_at=SUMMARY_TRIGGER_EARLY),
    "S4b keep-numbers, untrimmed": lambda: make_deep(**part2, summary_at=SUMMARY_TRIGGER,
                                                     summary_prompt=SUMMARY_PROMPT_KEEP),
}
S2B, S4B = "S2b 85%, untrimmed", "S4b keep-numbers, untrimmed"
print(list(ARMS_2))

In [ ]:
if RUN_PART2:
    for arm in ARMS_2:
        run_arm(arm, arms=ARMS_2, questions=QUESTIONS_2, recursion_limit=RECURSION_LIMIT_2)

In [ ]:
res2 = pd.concat([pd.DataFrame(load_cache(CACHE_DIR / f"{slug(a)}.jsonl").values()) for a in ARMS_2], ignore_index=True)
res2 = res2[res2.qid.isin(QUESTIONS_2.qid)]
res2["over_window"] = res2.n_over_window > 0
for name in ["get_series", "compact_conversation"]:
    res2[f"n_{name}"] = res2.tools.map(lambda t: t.get(name, 0))


def compactions_done(arm, qid):
    # Successful compact_conversation calls, read from the tool's own replies in the trace
    t = json.loads((TRACE_DIR / slug(arm) / f"{qid}.json").read_text())
    return sum(1 for m in t["messages"] if m["type"] == "tool" and m["data"].get("name") == "compact_conversation"
               and str(m["data"].get("content", "")).startswith("Conversation compacted"))


res2["n_compactions"] = [compactions_done(r.arm, r.qid) if r.n_compact_conversation else 0 for r in res2.itertuples()]
res2[["n_auto_compactions", "n_fallbacks"]] = [compactions(r.arm, r.qid) for r in res2.itertuples()]
res2 = res2.rename(columns={"correct": "correct_strict"}).assign(correct=lambda d: d.correct_strict)
res2["correct_lenient"] = res2.apply(lenient_ok, axis=1, questions=QUESTIONS_2)
res2["format_miss"] = res2.correct_lenient & ~res2.correct_strict
NEEDED = {r.qid: json.loads(r.needed) for r in QUESTIONS_2.itertuples()}


SERIES_HEADER_RE = re.compile(r"^(.+?) \| [a-z_0-9]+ \| \d{4}-\d{4}")   # first line of every get_series result
OTHER_NAMES_RE = {c: re.compile(r"\b(?:" + "|".join(re.escape(o) for o in sorted(POOL, key=len, reverse=True) if o != c)
                                + r")\b", re.I) for c in POOL}


def summarised_countries(t):
    # For each summariser call in a trace: the countries whose series were in the history it replaced. That's every
    # lookup before the cut, so a later summary must still carry what an earlier one kept. The cut is worked out
    # from the next agent call: it saw the summary plus the preserved messages, and its reply is the k-th AI message.
    ai_idx = [i for i, m in enumerate(t["messages"]) if m["type"] == "ai"]
    out, k = [], 0
    for j, c in enumerate(t["calls"]):
        if c["has_tools"]:
            k += 1
            continue
        nxt = next((x for x in t["calls"][j + 1:] if x["has_tools"]), None)
        if nxt is None or k >= len(ai_idx):
            out.append(None)   # the run ended before the summary was used
            continue
        cutoff = max(ai_idx[k] - (nxt["n_msgs"] - 1), 0)
        heads = (SERIES_HEADER_RE.match(str(m["data"]["content"])) for m in t["messages"][:cutoff] if m["type"] == "tool")
        out.append(sorted({h.group(1) for h in heads if h} & set(POOL)))
    return out


def near(text, country):
    # The text right after each mention of the country, up to the next country name or the end of the line,
    # so a value is only credited to the country it is written next to.
    for m in re.finditer(rf"\b{re.escape(country)}\b", text, re.I):   # \b so "Niger" doesn't match "Nigeria"
        rest = text[m.end():].split("\n")[0]
        stop = OTHER_NAMES_RE[country].search(rest)
        yield re.sub(r"(?<=\d),(?=\d{3})|[*`]", "", rest[:stop.start()] if stop else rest)


def fidelity(arm, qid):
    # One row per summary in this run: how many of the summarised countries kept a correct needed value,
    # stated only other numbers, or were dropped.
    t = json.loads((TRACE_DIR / slug(arm) / f"{qid}.json").read_text())
    texts = [c.get("text") for c in t["calls"] if not c["has_tools"]]
    rows = []
    for i, (countries, text) in enumerate(zip(summarised_countries(t), texts)):
        if countries is None or not text:
            continue
        kept = wrong = dropped = 0
        for country in (c for c in countries if c in NEEDED[qid]):   # ignore lookups the question didn't need
            segs = list(near(text, country))
            needed = {abs(float(v)) for v in NEEDED[qid][country]}
            # numbers written next to the country, not "CO2"; compared as numbers, so 9.3 matches 9.30
            nums = [abs(float(n)) for s in segs for n in re.findall(r"(?<![A-Za-z\d.])\d+(?:\.\d+)?", s)]
            if any(n in needed for n in nums):
                kept += 1
            elif any(not re.fullmatch(r"(?:19|20)\d\d", f"{n:g}") for n in nums):   # a number other than a year
                wrong += 1
            else:
                dropped += 1
        rows.append(dict(arm=arm, qid=qid, summary_no=i + 1, summarised=kept + wrong + dropped,
                         kept=kept, wrong=wrong, dropped=dropped))
    return rows


fid = pd.DataFrame([row for r in res2[res2.n_summary_calls > 0].itertuples() for row in fidelity(r.arm, r.qid)])
res2.drop(columns=["tools", "correct"]).to_csv("results_part2.csv", index=False)
fid.to_csv("summary_fidelity.csv", index=False)
print(res2.groupby("arm").status.value_counts().unstack(fill_value=0))

## Part 2 results

In [ ]:
g2 = res2.groupby("arm")
summary2 = pd.DataFrame({
    "accuracy": g2.correct_lenient.mean(),
    "acc_95ci": [f"{lo:.2f}-{hi:.2f}" for lo, hi in (wilson(d.correct_lenient.sum(), len(d)) for _, d in g2)],
    "accuracy_strict": g2.correct_strict.mean(),
    "runs_over_window": g2.over_window.mean(),
    "runs_with_summary": g2.n_summary_calls.apply(lambda s: (s > 0).mean()),
    "summaries_mean": g2.n_summary_calls.mean(),
    "auto_compactions_mean": g2.n_auto_compactions.mean(),
    "fallbacks_mean": g2.n_fallbacks.mean(),
    "compact_calls_mean": g2.n_compact_conversation.mean(),
    "compactions_mean": g2.n_compactions.mean(),
    "lookups_mean": g2.n_get_series.mean(),
    "peak_sent_median": g2.peak_sent.median(),
    "calls_mean": g2.n_calls.mean(),
    "wall_s_median": g2.wall_s.median(),
    "no_answer": g2.entity.apply(lambda s: s.isna().mean()),
}).reindex(list(ARMS_2)).round(3)
summary2.to_csv("summary_part2.csv")
by_level2 = res2.pivot_table(index="arm", columns="k", values="correct_lenient", aggfunc="mean").reindex(list(ARMS_2)).round(3)
by_level2.columns = [f"acc_{k}_countries" for k in by_level2.columns]
by_level2.to_csv("accuracy_by_level_part2.csv")
display(summary2)
display(by_level2)

if len(fid):
    fid_arm = fid.groupby("arm")[["summarised", "kept", "wrong", "dropped"]].sum()
    fid_arm["kept_share"] = (fid_arm.kept / fid_arm.summarised).round(3)
    fid_arm["wrong_share"] = (fid_arm.wrong / fid_arm.summarised).round(3)
    display(fid_arm.reindex([a for a in ARMS_2 if a in fid_arm.index]))
else:
    print("No summaries were produced in Part 2.")

base2 = res2[res2.arm == "S1 no summary"].set_index("qid").correct_lenient
for arm in [a for a in ARMS_2 if a != "S1 no summary"]:
    other = res2[res2.arm == arm].set_index("qid").correct_lenient.reindex(base2.index)
    b, c = int((base2 & ~other).sum()), int((~base2 & other).sum())
    p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    print(f"{arm} vs S1: wins {c}, losses {b}, exact McNemar p = {p:.3f}")

# How often each arm compacted, how many of those were the silent fallback, and what the summariser was given
summ_in = []
for r in res2.itertuples():
    t = json.loads((TRACE_DIR / slug(r.arm) / f"{r.qid}.json").read_text())
    calls = [c for c in t["calls"] if not c["has_tools"]]
    summ_in += [dict(arm=r.arm, sent=c["sent_tokens"], countries=len(cs or []))
                for c, cs in zip(calls, summarised_countries(t))]
summ_in = pd.DataFrame(summ_in)
compact2 = res2.groupby("arm")[["n_auto_compactions", "n_fallbacks", "n_summary_calls"]].sum().reindex(list(ARMS_2))
if len(summ_in):
    compact2 = compact2.join(summ_in.groupby("arm").agg(summariser_input_median=("sent", "median"),
                                                        countries_summarised=("countries", "sum")))
display(compact2)

In [ ]:
colors2 = dict(zip(ARMS_2, plt.cm.tab10.colors))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
TRIMMED_2 = {"S2 summary at 85%", "S3 summary at 50%", "S4 85% + keep-numbers prompt"}
for arm in ARMS_2:
    acc = res2[res2.arm == arm].groupby("k").correct_lenient.mean()
    trimmed = arm in TRIMMED_2   # the 4,000-token-trim arms, dashed
    ax.plot(acc.index, acc.values, marker="o", label=arm + (" (4k trim)" if trimmed else ""), color=colors2[arm],
            ls="--" if trimmed else "-", lw=1 if trimmed else 2, alpha=0.6 if trimmed else 1)
ax.set(xticks=list(LEVELS_2), xlabel="Countries the question needs (one lookup each)", ylabel="Accuracy (lenient)",
       ylim=(-0.05, 1.05), title="Summarisation: accuracy as lookups pile up")
ax.legend(fontsize=7)
ax = axes[1]
# Only arms whose summaries replaced at least one lookup; the trimmed arms' summaries never did
shares = fid_arm[fid_arm.summarised > 0] if len(fid) else pd.DataFrame()
shares = shares.reindex([a for a in ARMS_2 if a in shares.index])
if len(shares):
    left = np.zeros(len(shares))
    for col, colr in [("kept", "#2a9d8f"), ("wrong", "#e76f51"), ("dropped", "#adb5bd")]:
        vals = (shares[col] / shares.summarised).values
        ax.barh(range(len(shares)), vals, left=left, color=colr, label=col)
        left += vals
    ax.set_yticks(range(len(shares)), [f"{a}\n({int(n)} countries in {int(s)} summaries)" for a, n, s in
                                       zip(shares.index, shares.summarised,
                                           fid.groupby("arm").size().reindex(shares.index))], fontsize=8)
    ax.invert_yaxis()   # first arm on top
    ax.set(xlim=(0, 1), xlabel="Share of summarised countries", title="What summaries did with the numbers")
    ax.legend(fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
else:
    ax.text(0.5, 0.5, "no summary replaced a lookup", ha="center", va="center"); ax.set_axis_off()
fig.tight_layout(); fig.savefig(FIG_DIR / "summarisation.png", dpi=160); plt.show()

### Part 2 qualitative examples

The same question summarised by S2b (default prompt) and S4b (keep-the-numbers prompt), both untrimmed, then how S5 used `compact_conversation`.

In [ ]:
def summaries_of(arm, qid):
    t = json.loads((TRACE_DIR / slug(arm) / f"{qid}.json").read_text())
    return [c["text"] for c in t["calls"] if not c["has_tools"] and c.get("text")]


both = sorted(set(res2[(res2.arm == S2B) & (res2.n_summary_calls > 0)].qid)
              & set(res2[(res2.arm == S4B) & (res2.n_summary_calls > 0)].qid))
if both:
    qid = both[-1]
    print(QUESTIONS_2.set_index("qid").question[qid][:300], "\n")
    for arm in [S2B, S4B]:
        r = res2.set_index(["arm", "qid"]).loc[(arm, qid)]
        print(f"===== {arm}: correct={r.correct_lenient}, final: {(r.answer_text or '').strip().splitlines()[-1:]}")
        print(summaries_of(arm, qid)[0][:1500], "\n")
else:
    print("No question where both S2b and S4b summarised.")

s5 = res2[res2.arm == "S5 compact_conversation tool"]
print(f"S5: compact_conversation called in {(s5.n_compact_conversation > 0).sum()} of {len(s5)} runs, "
      f"{int(s5.n_compact_conversation.sum())} calls in total, {int(s5.n_compactions.sum())} actually compacted "
      f"(the rest were refused as too early). Summariser calls logged: {int(s5.n_summary_calls.sum())}.")
display(s5.groupby("k")[["n_compact_conversation", "n_compactions", "correct_lenient", "over_window"]].mean().round(2))

## Interpretation

### Part 1: big tool results (offloading)

- **Where the plain loop breaks:** exactly where the window does. It scored 13/16 with 1 or 3 country reports and 0/8 with 5, the runs that went over the window (a third of all runs, peaking around 21k tokens). Ollama silently drops part of the conversation, and the model answers without the data or the question. Nothing in the output says this happened.
- **What offloading recovers:** alone, it took accuracy from 13/24 to 18/24, and the 5-report questions from 0/8 to 4/8. The context stayed around 5–9k tokens. The model really used the files, with about 3.7 `read_file` and 2.3 `grep` calls per question, rather than guessing from the preview.
- **Whether summarisation helped on top: hard to say from Part 1.**
  - Arms 4 and 5 (4,000-token trim) compacted mostly through the silent fallback.
  - The untrimmed arms scored 22/24, but 4b summarised on only 2 of 24 questions. On the other 22 it sent the model the same thing as arm 3 and still scored 20 against 17. That gap is run-to-run noise, partly because these arms ran in a later session.
  - The cleanest signal is 5b: on the 8 questions where it summarised, it got 7 right, against arm 3's 4 on the same questions. That's suggestive, not conclusive.
- **Strict vs lenient:** much of the gap is format, not accuracy. With offloading, the model often stated the right answer in prose and dropped the `ANSWER:` line (10 of arm 3's 24 replies). More turns and a longer harness prompt made format-following worse.

### Part 2: many small results (summarisation)

- **S1 collapses at 50 countries:** 0/6. Two-thirds of those runs went over the window, and the rest gave up partway through the lookups. The `compact_conversation` tool (S5) didn't help: the model never called it in 18 runs, even with a system-prompt nudge.
- **The silent fallback:** with the constructor's 4,000-token trim (S2–S4), every compaction either summarised only the question or replaced the history with "too long to summarize". The summariser never saw a lookup, and 50-country accuracy stayed at 0/6.
- **Untrimmed, summarisation works.** On 50 countries, S2b and S3b got 3/6 and S4b 4/6. On the questions where a summary actually happened, S1 got 0–1 right, while S2b got 3/4, S3b 4/6 and S4b 5/5. Where no summary happened, the arms were roughly level with S1, as expected.
- **Do summaries keep the numbers?** They never stated a wrong number, but they dropped values: 34% of the countries they had to carry with the default prompt at 85% (S2b), 24% at 50% (S3b), and 16% with the keep-numbers prompt (S4b). One extra instruction roughly halved the losses. Dropped values cost time: at 50 countries, S2b made about 70 lookups per question as the model re-fetched what the summary had lost.
- **Earlier (50%) vs later (85%):** S3b summarised more often (22 summaries vs 12) and dropped less (24% vs 34%), with similar accuracy (13/18 vs 12/18). The direction is consistent, but each comparison rests on 4–6 summarised questions.

### Across both parts

- **When to use which:** offloading for big single results. It's cheap, it never loses data, and the agent can read the full result back. Summarisation for many small results that pile up, set up deliberately: an explicit trigger, a real token counter, no trim, a prompt that tells the summary to keep the numbers, and a check that it actually ran.
- **Cost:** offloading took the median from 16 s to 27 s per question and from 2 to 6 model calls. The summarisation arms took about 50 s per question in Part 2, similar to S1.
- **This harness with a small local model:** worth it when tool output can overflow the window, but only with the thresholds set for that window. Deep Agents' defaults suit frontier models: offloading at 20k tokens, and summarisation at 170k for models that don't report a context size, which includes every Ollama model. Out of the box, neither would fire. And don't rely on a 4B model to manage its own context through a tool.

## Limitations and next steps

- **Small sample:** 24 questions (8 per level) and one run per arm, so differences of a few questions are within noise; the confidence intervals and McNemar tests above say how much.
- **One model and one window size:** results for `qwen3.5:4b` at 16k may not transfer to larger models or longer windows. Rerunning with `MODEL = "qwen3.5:9b"` or a different `NUM_CTX` is a config change.
- **Scoring:** headline accuracy is lenient (the last line must name the gold answer and number). Strict accuracy, which also needs the `ANSWER:` line, is in the tables. The gap between them is mostly format-following, which got worse as the Deep Agents harness grew the prompt and the number of turns. The 1-country "peak year" questions ask for the year in the name slot, and models often put the country there instead; that question wording penalises every arm the same way.
- **Run-to-run noise:** Ollama isn't fully deterministic at temperature 0. Arms 3 and 4 were effectively identical in the first run and still disagreed on 3 of 24 questions, so treat gaps of 2–3 questions as noise.
- **Synthetic task:** the questions are generated from one dataset and need exact number lookups, which favours offloading (the raw data is recoverable) over summarisation (a summary can paraphrase numbers away).
- **Part 2 is small:** 18 questions (6 per level), so only large differences mean much. In the first run, S2 and S4 disagreed with S1 on 2 of the 30-country questions where neither had compacted anything, which shows the noise level. The fidelity check is a heuristic: it looks for the needed value right after the country's name, so a correct value written differently (e.g. "about 1.2 thousand") counts as wrong, and a country written another way (e.g. "UAE") or a value on a later line counts as dropped. Which history a summary replaced is worked out from the message counts in the trace.
- **The trimmed arms were a setup mistake, kept on purpose:** arms 4, 5 and S2–S4 built the summariser with its constructor default `trim_tokens_to_summarize=4000`, not `create_deep_agent`'s `None`. They show what that pitfall does; the "b" arms are the fair test of summarisation. The b arms ran in a later session than the arms they're compared with, so run-to-run noise applies.
- **Compaction and fallback counts are inferred** from the number of messages each agent call saw, not read from the middleware, since Deep Agents keeps the summary in private state.
- **The summariser's trigger counts messages and the system prompt, not the tool schemas** (that's how `count_tokens` is written), so it fires slightly later than the window share suggests.
- **Thresholds are mine:** offloading at 4,000 characters and summarising at 85% of the window are tuned for this window, not Deep Agents' defaults. With the defaults, neither technique would fire for this model.
- **Token counts** use the Qwen tokenizer on message text and leave out the chat template and tool schemas, so they slightly undercount what the model received.
- **Not tested here:** skills, long-term memory and the subagent arm (optional, `RUN_SUBAGENT_ARM = True`), which is LangChain's other answer to noisy lookups.

## Environment

In [ ]:
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["ollama", "show", MODEL, "--modelfile"], capture_output=True, text=True).stdout.splitlines()[:3])
!pip freeze | grep -iE "^(deepagents|langchain|langchain-core|langchain-ollama|langgraph|langgraph-checkpoint|langgraph-prebuilt|ollama|tokenizers|pandas|scipy)=="

## Download all results

This cell zips the figures, the CSVs and the per-run caches and traces, and downloads the zip on Colab.

In [ ]:
import zipfile
from datetime import datetime

zip_path = Path(f"context_rot_results_{datetime.now():%Y%m%d_%H%M}.zip")
files = [*FIG_DIR.glob("*.png"), *(Path(f) for f in ["questions.csv", "results.csv", "summary.csv", "accuracy_by_level.csv",
                                                     "questions_part2.csv", "results_part2.csv", "summary_part2.csv",
                                                     "accuracy_by_level_part2.csv", "summary_fidelity.csv"])]
files += [*CACHE_DIR.glob("*.jsonl"), *TRACE_DIR.rglob("*.json")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if f.exists():
            z.write(f, f.relative_to(BASE_DIR) if BASE_DIR in f.parents else f)
print(f"{zip_path}: {len(z.namelist())} files, {zip_path.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("Not on Colab: the zip is next to the notebook.")